In [1]:
import sys

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
sys.path.append("..")

In [4]:
from module1.ingest import load_faq_data

In [5]:
documents = load_faq_data()

In [6]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [7]:
documents = documents_llm

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [13]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [14]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [15]:
response.output_parsed.questions

['I just found this course — is it still possible to enroll now?',
 'Can I join the class late, or am I too late already?',
 'If I start the course after it began, can I still take part?',
 'Is it okay to begin this course now, or do I have to wait for the next run?',
 'I missed the start date — can I still join and get a certificate?']

In [16]:
from evaluation_utils import llm_structured, llm_structured_retry

In [17]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it still possible to join now?', 'If I start late, am I still eligible for the course certificate?', 'Do I need to finish and submit the project before submissions close to get a certificate?', 'Can I enroll after the course has already started, or is it too late?', 'What’s the deadline for submitting the project if I want the certificate?']


In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it still possible to join now?',
  'document': '74eb249bbf'},
 {'question': 'If I start late, am I still eligible for the course certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish and submit the project before submissions close to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll after the course has already started, or is it too late?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for submitting the project if I want the certificate?',
  'document': '74eb249bbf'}]

In [19]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [20]:
from tqdm import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:10<00:00,  2.08s/it]


In [21]:
ground_truth[:5]

[{'question': 'Can I still start the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to join llm-zoomcamp now?',
  'document': '74eb249bbf'},
 {'question': 'I just came across the course—am I still able to enroll?',
  'document': '74eb249bbf'},
 {'question': 'Can new students join even after the course has started?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get the certificate?',
  'document': '74eb249bbf'}]

In [22]:
len(ground_truth)

25

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

100%|██████████| 103/103 [00:32<00:00,  3.20it/s]


In [25]:
len(results)

103

In [26]:
results[0]

([{'question': 'Is it still okay to join this course if I found it late?',
   'document': '74eb249bbf'},
  {'question': 'Can I enroll after the course has already started?',
   'document': '74eb249bbf'},
  {'question': 'If I join the course now, can I still get a certificate?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to finish the project before submissions close to get certified?',
   'document': '74eb249bbf'},
  {'question': 'What’s the deadline for submitting the project if I want the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=289))

In [27]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [28]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)